In [2]:
# =========================================================
# FAVAR Connectedness for merged_var_input.csv
# - Input: ./merged_var_input.csv
# - Uses transformed/stationary columns
# - PCA-based latent factors
# - VAR on [observed vars + factors]
# - Generalized FEVD / connectedness summary
# =========================================================

import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.api import VAR

# =========================================================
# 0. Config
# =========================================================
DATA_PATH = "./merged_var_input.csv"
OUT_DIR = "./output_favar"

DATE_COL = "Date"
OBS_COLS = ["dlog_SOLVPN", "dlog_COPPER", "dlog_DXY", "d_UST10Y", "dlog_VIX"]

# factor settings
N_FACTORS = 2     # 현재 5변수라 우선 2개 권장
P = 1             # VAR lag
H = 10            # FEVD horizon

os.makedirs(OUT_DIR, exist_ok=True)

# =========================================================
# 1. Load data
# =========================================================
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"File not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

required_cols = [DATE_COL] + OBS_COLS
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL).reset_index(drop=True)
df = df[required_cols].dropna().reset_index(drop=True)

dates = df[DATE_COL].copy()
X_obs = df[OBS_COLS].copy()

print("Loaded shape:", X_obs.shape)
print("Columns:", OBS_COLS)
print("Date range:", dates.min(), "~", dates.max())

# Save cleaned input
df.to_csv(os.path.join(OUT_DIR, "favar_input_clean.csv"), index=False)

# =========================================================
# 2. Standardize and extract factors via PCA
# =========================================================
scaler = StandardScaler()
X_std = scaler.fit_transform(X_obs)

pca = PCA(n_components=N_FACTORS, random_state=42)
factors = pca.fit_transform(X_std)

factor_cols = [f"Factor{i+1}" for i in range(N_FACTORS)]
factor_df = pd.DataFrame(factors, columns=factor_cols, index=df.index)

# factor loadings
loadings = pd.DataFrame(
    pca.components_.T,
    index=OBS_COLS,
    columns=factor_cols
)

explained_var = pd.DataFrame({
    "Factor": factor_cols,
    "ExplainedVarianceRatio": pca.explained_variance_ratio_,
    "CumulativeExplainedVariance": np.cumsum(pca.explained_variance_ratio_)
})

# Save factor outputs
factor_df_with_date = pd.concat([dates, factor_df], axis=1)
factor_df_with_date.to_csv(os.path.join(OUT_DIR, "factors_timeseries.csv"), index=False)
loadings.to_csv(os.path.join(OUT_DIR, "factor_loadings.csv"))
explained_var.to_csv(os.path.join(OUT_DIR, "factor_explained_variance.csv"), index=False)

print("\nExplained variance ratio")
print(explained_var)

print("\nFactor loadings")
print(loadings.round(4))

# =========================================================
# 3. Build FAVAR dataset
# =========================================================
favar_df = pd.concat([dates, X_obs, factor_df], axis=1)
ALL_COLS = OBS_COLS + factor_cols

favar_df.to_csv(os.path.join(OUT_DIR, "favar_dataset.csv"), index=False)

# =========================================================
# 4. Fit VAR on [observed vars + factors]
# =========================================================
model = VAR(favar_df[ALL_COLS])
res = model.fit(P)

# Save text summary
with open(os.path.join(OUT_DIR, "favar_var_summary.txt"), "w", encoding="utf-8") as f:
    f.write(str(res.summary()))

print("\nVAR fitted.")
print("Lag order:", res.k_ar)

# Save coefficient matrices
for lag_idx in range(res.k_ar):
    coef_df = pd.DataFrame(
        res.coefs[lag_idx],
        index=ALL_COLS,
        columns=ALL_COLS
    )
    coef_df.to_csv(os.path.join(OUT_DIR, f"favar_A{lag_idx+1}.csv"))

sigma_u_df = pd.DataFrame(res.sigma_u, index=ALL_COLS, columns=ALL_COLS)
sigma_u_df.to_csv(os.path.join(OUT_DIR, "favar_residual_covariance.csv"))

# =========================================================
# 5. Generalized FEVD helper
# =========================================================
def var_companion(A_list):
    p = len(A_list)
    k = A_list[0].shape[0]
    top = np.hstack(A_list)
    if p == 1:
        return top
    bottom = np.hstack([np.eye(k * (p - 1)), np.zeros((k * (p - 1), k))])
    return np.vstack([top, bottom])

def generalized_fevd(A_list, Sigma, H):
    """
    Pesaran-Shin generalized FEVD
    A_list: [A1, ..., Ap], each (k,k)
    Sigma: residual covariance matrix (k,k)
    Returns row-normalized FEVD matrix
    """
    k = Sigma.shape[0]
    p = len(A_list)

    F = var_companion(A_list)
    kp = k * p if p > 1 else k
    J = np.hstack([np.eye(k), np.zeros((k, kp - k))]) if p > 1 else np.eye(k)

    Phi = []
    F_power = np.eye(kp)
    for h in range(H):
        Phi_h = J @ F_power @ J.T
        Phi.append(Phi_h)
        F_power = F_power @ F

    theta = np.zeros((k, k))
    sigma_diag = np.diag(Sigma).copy()
    sigma_diag[sigma_diag <= 1e-12] = 1e-12

    for i in range(k):
        for j in range(k):
            e_i = np.zeros((k, 1)); e_i[i, 0] = 1.0
            e_j = np.zeros((k, 1)); e_j[j, 0] = 1.0

            num = 0.0
            den = 0.0
            for h in range(H):
                phi_h = Phi[h]
                num += float((e_i.T @ phi_h @ Sigma @ e_j) ** 2 / sigma_diag[j])
                den += float(e_i.T @ phi_h @ Sigma @ phi_h.T @ e_i)

            theta[i, j] = num / max(den, 1e-12)

    row_sums = theta.sum(axis=1, keepdims=True)
    row_sums[row_sums <= 1e-12] = 1e-12
    theta_norm = theta / row_sums
    return theta_norm

def spillover_from_fevd(theta):
    k = theta.shape[0]
    total = 100.0 * (theta.sum() - np.trace(theta)) / k
    to_others = 100.0 * (theta.sum(axis=0) - np.diag(theta))
    from_others = 100.0 * (theta.sum(axis=1) - np.diag(theta))
    net = to_others - from_others
    return total, to_others, from_others, net

# =========================================================
# 6. Compute generalized FEVD and connectedness
# =========================================================
A_list = [res.coefs[i] for i in range(res.k_ar)]
Sigma = np.asarray(res.sigma_u)

theta = generalized_fevd(A_list, Sigma, H)
total, to_, from_, net_ = spillover_from_fevd(theta)

theta_df = pd.DataFrame(theta, index=ALL_COLS, columns=ALL_COLS)
theta_df.to_csv(os.path.join(OUT_DIR, "favar_fevd_matrix.csv"))

summary_rows = []
for i, var in enumerate(ALL_COLS):
    summary_rows.append({
        "Variable": var,
        "TO": float(to_[i]),
        "FROM": float(from_[i]),
        "NET": float(net_[i])
    })

conn_summary_df = pd.DataFrame(summary_rows)
conn_summary_df.to_csv(os.path.join(OUT_DIR, "favar_connectedness_summary.csv"), index=False)

# observed-only FEVD submatrix
obs_theta_df = theta_df.loc[OBS_COLS, OBS_COLS]
obs_theta_df.to_csv(os.path.join(OUT_DIR, "favar_observed_only_fevd_matrix.csv"))

# pairwise long format
pairwise_records = []
for i, response in enumerate(ALL_COLS):
    for j, shock in enumerate(ALL_COLS):
        pairwise_records.append({
            "Response": response,
            "Shock": shock,
            "FEVD": float(theta[i, j])
        })

pairwise_df = pd.DataFrame(pairwise_records)
pairwise_df.to_csv(os.path.join(OUT_DIR, "favar_pairwise_fevd_long.csv"), index=False)

# =========================================================
# 7. Observed-only connectedness summary
# =========================================================
obs_idx = [ALL_COLS.index(c) for c in OBS_COLS]
theta_obs = theta[np.ix_(obs_idx, obs_idx)]

# row-normalize observed-only submatrix again for observed-only comparison
row_sum_obs = theta_obs.sum(axis=1, keepdims=True)
row_sum_obs[row_sum_obs <= 1e-12] = 1e-12
theta_obs_norm = theta_obs / row_sum_obs

obs_total, obs_to, obs_from, obs_net = spillover_from_fevd(theta_obs_norm)

obs_summary_rows = []
for i, var in enumerate(OBS_COLS):
    obs_summary_rows.append({
        "Variable": var,
        "TO": float(obs_to[i]),
        "FROM": float(obs_from[i]),
        "NET": float(obs_net[i])
    })

obs_conn_summary_df = pd.DataFrame(obs_summary_rows)
obs_conn_summary_df.to_csv(os.path.join(OUT_DIR, "favar_observed_only_connectedness_summary.csv"), index=False)

# =========================================================
# 8. Metadata
# =========================================================
meta = {
    "model": "FAVAR Connectedness (PCA-based)",
    "data_path": DATA_PATH,
    "date_col": DATE_COL,
    "observed_cols": OBS_COLS,
    "factor_cols": factor_cols,
    "n_factors": N_FACTORS,
    "lag_order": P,
    "fevd_horizon": H,
    "explained_variance_ratio": pca.explained_variance_ratio_.tolist(),
    "cumulative_explained_variance": np.cumsum(pca.explained_variance_ratio_).tolist(),
    "total_spillover_all": float(total),
    "total_spillover_observed_only": float(obs_total),
    "n_obs_used": int(len(favar_df))
}
with open(os.path.join(OUT_DIR, "metadata.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2, default=str)

# =========================================================
# 9. Plots
# =========================================================
# (1) Explained variance
plt.figure(figsize=(8, 5))
plt.bar(explained_var["Factor"], explained_var["ExplainedVarianceRatio"])
plt.title("FAVAR: Explained Variance by Factor")
plt.xlabel("Factor")
plt.ylabel("Explained Variance Ratio")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "factor_explained_variance.png"), dpi=300)
plt.show()

# (2) Factor time series
plt.figure(figsize=(12, 5))
for c in factor_cols:
    plt.plot(dates, factor_df[c], label=c)
plt.title("FAVAR: Factor Time Series")
plt.xlabel("Date")
plt.ylabel("Factor Value")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "factors_timeseries.png"), dpi=300)
plt.show()

# (3) FEVD heatmap
plt.figure(figsize=(8, 6))
plt.imshow(theta_df.values, aspect="auto")
plt.xticks(range(len(ALL_COLS)), ALL_COLS, rotation=45, ha="right")
plt.yticks(range(len(ALL_COLS)), ALL_COLS)
plt.title("FAVAR Generalized FEVD Matrix")
plt.colorbar()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "favar_fevd_heatmap.png"), dpi=300)
plt.show()

# (4) Observed-only FEVD heatmap
plt.figure(figsize=(7, 5))
plt.imshow(obs_theta_df.values, aspect="auto")
plt.xticks(range(len(OBS_COLS)), OBS_COLS, rotation=45, ha="right")
plt.yticks(range(len(OBS_COLS)), OBS_COLS)
plt.title("FAVAR Observed-only FEVD Submatrix")
plt.colorbar()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "favar_observed_only_fevd_heatmap.png"), dpi=300)
plt.show()

# =========================================================
# 10. Print summary
# =========================================================
print("\n[Done] FAVAR completed.")
print("Output folder:", OUT_DIR)

print("\nExplained variance")
print(explained_var)

print("\nConnectedness summary (all variables incl. factors)")
print(conn_summary_df.round(4))

print("\nObserved-only connectedness summary")
print(obs_conn_summary_df.round(4))

print(f"\nTotal spillover (all variables incl. factors): {total:.4f}")
print(f"Total spillover (observed-only renormalized): {obs_total:.4f}")

Loaded shape: (1325, 6)
Columns: ['dlog_SOLVPN', 'dlog_COPPER', 'dlog_DXY', 'd_UST10Y', 'dlog_VIX']
Date range: 2020-10-13 00:00:00 ~ 2026-01-12 00:00:00
Usable observations after lagging: 1324
Rolling windows to estimate: 1125

========== Running tau=0.05 ==========

========== Running tau=0.5 ==========

========== Running tau=0.95 ==========


KeyboardInterrupt: 